# 7-Qubit Superconducting Quantum Processor Design Suite
## Complete Lecture Implementation (Simon's Quantum Architecture)

This notebook implements the complete 7-qubit quantum processor design from Simon's Qiskit Metal lecture series:
- **7 Transmon Qubits:** Central transmon `Q4` with charge line (`TransmonPocketCL`) + 6 outer transmons (`TransmonPocket6`).
- **6 Wirebond Launchpads:** 4 at the top ($y = 4.0\text{--}4.2\text{ mm}$) and 2 at the bottom ($y = -2.0\text{ mm}$).
- **6 Meandered Readout Resonators:** Detuned coplanar waveguide (CPW) lines for individual qubit state readout.
- **6 Inter-Qubit Coupling Buses:** Nearest-neighbor entanglement coupling lines.
- **Nanofabrication Mask Export:** Export to GDSII format for KLayout inspection.

In [3]:
# Cell 0: Load extensions and PyQt GUI backend
%reload_ext autoreload
%autoreload 2
%gui qt

import qiskit_metal as metal
from qiskit_metal import designs, draw, MetalGUI, Dict
from qiskit_metal.qlibrary.qubits.transmon_pocket_6 import TransmonPocket6
from qiskit_metal.qlibrary.qubits.transmon_pocket_cl import TransmonPocketCL
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
import pyEPR as epr

In [4]:
# Cell 1: Initialize 10mm x 10mm Planar Silicon Chip Design
design = designs.DesignPlanar({}, True)
design.chips.main.size['size_x'] = '10mm'
design.chips.main.size['size_y'] = '10mm'
design.chips.main.size['center_x'] = '0mm'
design.chips.main.size['center_y'] = '0mm'

# Open the interactive GUI canvas
gui = MetalGUI(design)
gui.autoscale()
gui.rebuild()

In [5]:
# Cell 2: Place 7 Transmons (Matching Simon's Exact Layout)
# Center Transmon with Charge Line (Q4)
q4_opts = Dict(
    pos_x="0.0mm",
    pos_y="0.87mm",
    orientation="0.0",
    pad_width="455um",
    pad_height="90um",
    pad_gap="30um",
    pocket_width="650um",
    pocket_height="650um",
    make_CL=True,
    cl_gap="6um",
    cl_width="10um",
    cl_length="20um",
    cl_ground_gap="6um",
    cl_pocket_edge="50",
    cl_off_center="0um",
    connection_pads=Dict(
        bus_left=Dict(loc_W=-1, loc_H=-1, pad_width='50um', pad_height='30um'),
        bus_right=Dict(loc_W=1, loc_H=-1, pad_width='50um', pad_height='30um')
    )
)
q4 = TransmonPocketCL(design, "Q4", options=q4_opts)

# Outer 6 Transmons
other_qubits = {
    "Q0": Dict(pos_x="-2.5mm", pos_y="0.0mm", orientation="90.0",
               connection_pads=Dict(bus_top=Dict(loc_W=1, loc_H=1), readout=Dict(loc_W=-1, loc_H=-1))),
    "Q1": Dict(pos_x="-2.5mm", pos_y="1.8mm", orientation="90.0",
               connection_pads=Dict(bus_bot=Dict(loc_W=1, loc_H=-1), bus_right=Dict(loc_W=-1, loc_H=-1), readout=Dict(loc_W=1, loc_H=1))),
    "Q3": Dict(pos_x="-1.35mm", pos_y="0.87mm", orientation="0.0",
               connection_pads=Dict(bus_left=Dict(loc_W=-1, loc_H=1), bus_right=Dict(loc_W=1, loc_H=-1), readout=Dict(loc_W=1, loc_H=1))),
    "Q5": Dict(pos_x="1.35mm", pos_y="0.87mm", orientation="0.0",
               connection_pads=Dict(bus_left=Dict(loc_W=-1, loc_H=-1), bus_right=Dict(loc_W=1, loc_H=1), readout=Dict(loc_W=-1, loc_H=1))),
    "Q2": Dict(pos_x="2.5mm", pos_y="1.8mm", orientation="90.0",
               connection_pads=Dict(bus_bot=Dict(loc_W=1, loc_H=-1), bus_left=Dict(loc_W=-1, loc_H=-1), readout=Dict(loc_W=1, loc_H=1))),
    "Q6": Dict(pos_x="2.5mm", pos_y="0.0mm", orientation="90.0",
               connection_pads=Dict(bus_top=Dict(loc_W=1, loc_H=1), readout=Dict(loc_W=-1, loc_H=-1))),
}

for name, opts in other_qubits.items():
    opts.update(dict(
        pad_width="455um",
        pad_height="90um",
        pad_gap="30um",
        pocket_width="650um",
        pocket_height="650um"
    ))
    TransmonPocket6(design, name, options=opts)

design.rebuild()
gui.autoscale()
print("Placed all 7 Transmons successfully!")

Placed all 7 Transmons successfully!


In [6]:
# Cell 3: Place 6 Wirebond Launchpads (4 Top + 2 Bottom)
launchpads = {
    "LP_Q1": Dict(pos_x="-2.5mm", pos_y="4.0mm", orientation="-90.0"),
    "LP_Q3": Dict(pos_x="-1.0mm", pos_y="4.2mm", orientation="-90.0"),
    "LP_Q5": Dict(pos_x="1.0mm",  pos_y="4.2mm", orientation="-90.0"),
    "LP_Q2": Dict(pos_x="2.5mm",  pos_y="4.0mm", orientation="-90.0"),
    "LP_Q0": Dict(pos_x="-2.5mm", pos_y="-2.0mm", orientation="90.0"),
    "LP_Q6": Dict(pos_x="2.5mm",  pos_y="-2.0mm", orientation="90.0"),
}

for name, opts in launchpads.items():
    LaunchpadWirebond(design, name, options=opts)

design.rebuild()
gui.autoscale()
print("Placed all 6 Wirebond Launchpads successfully!")

Placed all 6 Wirebond Launchpads successfully!


In [7]:
# Cell 4: Route 6 Meandered Readout Resonators
readouts = [
    ("Readout_Q0", "Q0", "readout", "LP_Q0", "tie"),
    ("Readout_Q1", "Q1", "readout", "LP_Q1", "tie"),
    ("Readout_Q3", "Q3", "readout", "LP_Q3", "tie"),
    ("Readout_Q5", "Q5", "readout", "LP_Q5", "tie"),
    ("Readout_Q2", "Q2", "readout", "LP_Q2", "tie"),
    ("Readout_Q6", "Q6", "readout", "LP_Q6", "tie"),
]

for name, q_comp, q_pin, lp_comp, lp_pin in readouts:
    ro_opts = Dict(
        pin_inputs=Dict(
            start_pin=Dict(component=q_comp, pin=q_pin),
            end_pin=Dict(component=lp_comp, pin=lp_pin)
        ),
        fillet="99um"
    )
    RouteMeander(design, name, options=ro_opts)

design.rebuild()
gui.autoscale()
print("Routed all 6 Meandered Readout Resonators successfully!")

12:15AM 58s WARNING [check_lengths]: For path table, component=Readout_Q0, key=trace has short segments that could cause issues with fillet. Values in (1-1)  are index(es) in shapely geometry.
  self.tables[kind] = pd.concat([table, df],

12:15AM 59s WARNING [check_lengths]: For path table, component=Readout_Q0, key=cut has short segments that could cause issues with fillet. Values in (1-1)  are index(es) in shapely geometry.
12:15AM 59s WARNING [check_lengths]: For path table, component=Readout_Q1, key=trace has short segments that could cause issues with fillet. Values in (1-1)  are index(es) in shapely geometry.
12:15AM 59s WARNING [check_lengths]: For path table, component=Readout_Q1, key=cut has short segments that could cause issues with fillet. Values in (1-1)  are index(es) in shapely geometry.
12:15AM 59s WARNING [check_lengths]: For path table, component=Readout_Q3, key=trace has short segments that could cause issues with fillet. Values in (1-3) (28-29)  are index(es) in sha

Routed all 6 Meandered Readout Resonators successfully!


In [8]:
# Cell 5: Route 6 Inter-Qubit Coupling Buses
coupling_buses = [
    ("Bus_Q0_Q1", "Q0", "bus_top",   "Q1", "bus_bot"),
    ("Bus_Q1_Q3", "Q1", "bus_right", "Q3", "bus_left"),
    ("Bus_Q3_Q4", "Q3", "bus_right", "Q4", "bus_left"),
    ("Bus_Q4_Q5", "Q4", "bus_right", "Q5", "bus_left"),
    ("Bus_Q5_Q2", "Q5", "bus_right", "Q2", "bus_left"),
    ("Bus_Q2_Q6", "Q2", "bus_bot",   "Q6", "bus_top"),
]

for name, q_start, pin_start, q_end, pin_end in coupling_buses:
    bus_opts = Dict(
        pin_inputs=Dict(
            start_pin=Dict(component=q_start, pin=pin_start),
            end_pin=Dict(component=q_end, pin=pin_end)
        ),
        fillet="99um"
    )
    RouteMeander(design, name, options=bus_opts)

design.rebuild()
gui.autoscale()
print("Routed all 6 Inter-Qubit Coupling Buses successfully!")

12:16AM 01s WARNING [check_lengths]: For path table, component=Bus_Q0_Q1, key=trace has short segments that could cause issues with fillet. Values in (1-1) (20-20)  are index(es) in shapely geometry.
12:16AM 01s WARNING [check_lengths]: For path table, component=Bus_Q0_Q1, key=cut has short segments that could cause issues with fillet. Values in (1-1) (20-20)  are index(es) in shapely geometry.
12:16AM 01s WARNING [check_lengths]: For path table, component=Bus_Q3_Q4, key=trace has short segments that could cause issues with fillet. Values in (1-1)  are index(es) in shapely geometry.
12:16AM 01s WARNING [check_lengths]: For path table, component=Bus_Q3_Q4, key=cut has short segments that could cause issues with fillet. Values in (1-1)  are index(es) in shapely geometry.
12:16AM 01s WARNING [check_lengths]: For path table, component=Bus_Q4_Q5, key=trace has short segments that could cause issues with fillet. Values in (1-1)  are index(es) in shapely geometry.
12:16AM 01s WARNING [check_l

Routed all 6 Inter-Qubit Coupling Buses successfully!


In [9]:
# Cell 6: Export Nanofabrication Mask (GDSII)
a_gds = design.renderers.gds
a_gds.export_to_gds("seven_qubit_chip_mask.gds")
print("Exported nanofabrication mask to seven_qubit_chip_mask.gds for KLayout!")

  pd.concat(copy_subtract, ignore_index=False))

  pd.concat(copy_no_subtract, ignore_index=False))

12:16AM 04s WARNING [_import_junction_gds_file]: Not able to find file:"../resources/Fake_Junctions.GDS".  Not used to replace junction. Checked directory:"C:\Users\noobg\Desktop\resources".


Exported nanofabrication mask to seven_qubit_chip_mask.gds for KLayout!
